In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.ndimage import affine_transform
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.spatial.transform import Rotation as R

# === CONFIGURATION ===
DATA_DIR = "/home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data"
print(f'Data directory: {DATA_DIR}')
print('Imports OK')

Data directory: /home/tim-external/ros_ws/src/fsregistration/plotting_results/2d/data
Imports OK


In [2]:
def load_csv_array(filepath):
    if not os.path.exists(filepath):
        return None
    return np.loadtxt(filepath)


def load_meta_csv(filepath):
    if not os.path.exists(filepath):
        return None
    return pd.read_csv(filepath)


# Load data
img1 = load_csv_array(os.path.join(DATA_DIR, 'input1.csv'))
img2 = load_csv_array(os.path.join(DATA_DIR, 'input2.csv'))
meta_df = load_meta_csv(os.path.join(DATA_DIR, 'registration_meta.csv'))

if img1 is None or img2 is None or meta_df is None:
    print('ERROR: Missing data files. Make sure viewBoreasPairs.py has been run.')
    print(f'  input1.csv exists: {os.path.exists(os.path.join(DATA_DIR, "input1.csv"))}')
    print(f'  input2.csv exists: {os.path.exists(os.path.join(DATA_DIR, "input2.csv"))}')
    print(f'  registration_meta.csv exists: {os.path.exists(os.path.join(DATA_DIR, "registration_meta.csv"))}')
else:
    meta = meta_df.iloc[0]
    N = int(meta['N'])
    rot_deg = meta['rot_angle_deg']
    tx_m = meta['tx_m']
    ty_m = meta['ty_m']
    confidence = meta['confidence']
    time_ms = meta['time_ms']
    gt_rot_err = meta['gt_rot_err_deg']
    gt_trans_err = meta['gt_trans_err_m']
    frame1 = int(meta['frame1'])
    frame2 = int(meta['frame2'])
    n_solutions = int(meta['n_solutions'])
    pixel_size = meta['pixel_size_m']
    
    print(f'Loaded pair: frames {frame1}->{frame2}')
    print(f'Image size: {N}x{N}')
    print(f'Pixel size: {pixel_size} m')
    print(f'Rotation: {rot_deg:.2f} deg')
    print(f'Translation: tx={tx_m:.2f} m, ty={ty_m:.2f} m')
    print(f'Confidence: {confidence:.3f}')
    print(f'GT Error: rot={gt_rot_err:.2f} deg, trans={gt_trans_err:.2f} m')
    print(f'Solutions: {n_solutions}')
    print()
    print(meta_df.to_string())

ERROR: Missing data files. Make sure viewBoreasPairs.py has been run.
  input1.csv exists: False
  input2.csv exists: False
  registration_meta.csv exists: False


## Blended Registration Result

The second image is warped using the estimated transformation (rotation + translation in SI units, converted to pixels using `pixel_size_m`), then blended 50/50 with the first image. This shows how well the registration aligns the two scans.

In [3]:
if img1 is not None and img2 is not None and meta_df is not None:
    # Convert SI translation to pixel shifts
    tx_px = tx_m / pixel_size
    ty_px = ty_m / pixel_size
    rot_rad = np.deg2rad(rot_deg)
    
    # 2D rotation matrix
    c, s = np.cos(rot_rad), np.sin(rot_rad)
    R_mat = np.array([[c, -s], [s, c]])
    
    # Warp img2 using backward affine transform (same convention as existing notebook)
    h, w = img2.shape
    c_rc = np.array([(N - 1) / 2.0, (N - 1) / 2.0])  # pixel center
    t_rc = np.array([ty_px, tx_px])  # (row_offset, col_offset) = (y, x)
    offset = c_rc - R_mat.T @ c_rc + t_rc
    
    transformed = affine_transform(
        input=img2,
        matrix=R_mat.T,
        offset=offset,
        output_shape=(h, w),
        mode='constant',
        cval=0.0,
        order=1
    )
    
    # Blend 50/50
    blended = 0.5 * img1 + 0.5 * transformed
    
    # Normalize for display
    img1_norm = (img1 - img1.min()) / (img1.max() - img1.min() + 1e-10)
    img2_norm = (img2 - img2.min()) / (img2.max() - img2.min() + 1e-10)
    blended_norm = (blended - blended.min()) / (blended.max() - blended.min() + 1e-10)
    
    # Title text
    title = (f'Pair (frames {frame1}->{frame2})<br>'
             f'Rot: {rot_deg:.2f} deg | Tx: {tx_m:.2f} m | Ty: {ty_m:.2f} m<br>'
             f'Confidence: {confidence:.3f} | Time: {time_ms:.1f} ms | Solutions: {n_solutions}<br>'
             f'GT Error: Rot={gt_rot_err:.2f} deg | Trans={gt_trans_err:.2f} m')
    
    # Create subplots: top row = img1 + img2, bottom row = blended
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=('Input Image 1', 'Input Image 2', 'Blended (warped overlay)', ''),
        specs=[[{'type': 'heatmap'}, {'type': 'heatmap'}],
               [{'type': 'heatmap', 'colspan': 2}, {'type': 'none'}]],
        vertical_spacing=0.08,
        horizontal_spacing=0.08
    )
    
    fig.add_trace(go.Heatmap(z=img1_norm, colorscale='Viridis', showscale=True,
                             colorbar_title='Intensity'), row=1, col=1)
    fig.add_trace(go.Heatmap(z=img2_norm, colorscale='Viridis', showscale=False), row=1, col=2)
    fig.add_trace(go.Heatmap(z=blended_norm, colorscale='Viridis', showscale=False), row=2, col=1)
    
    fig.update_layout(
        height=900,
        width=1200,
        title_text=title,
        title_font_size=16,
        title_x=0.5,
        title_y=0.98
    )
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(showticklabels=False, showgrid=False)
    
    fig.show()
else:
    print('Cannot display: missing data files.')

Cannot display: missing data files.
